## Visualize SDES Stock Distribution

**Purpose**
Builds descriptive plots and summary tables from the aggregated SDES stock dataset.

**Primary inputs**
- `data/derived/building_stock/building_stock_sdes2018_aggregated.csv`
- `project/input/resources_data.csv`

**Primary outputs**
- `data/derived/building_stock/img/*.png`
- `data/derived/building_stock/img/stock_income_performance_table.csv`



## Execution Notes

- Run cells top-to-bottom from a clean kernel.
- Paths are notebook-local and follow the refactored domain layout (`data/` for inputs and small derived tables, `runs/` for generated scenario outputs).
- If you switch datasets or scenarios, update the path/config variables in the first code cells before running downstream steps.



# Step 2: SDES Stock Visualization

**Purpose:** Visualize the processed SDES building stock — distribution by performance, energy, income.

**Project imports:** `project.model`, `project.utils`, `project.input.resources`.

**Inputs:** building_stock_sdes2018_aggregated.csv (from Step 1).

**Outputs:** PNG visualizations, grouped CSV tables.


In [ ]:
import os
import pandas as pd

from project.input.resources import resources_data
from project.model import get_inputs
from project.utils import subplots_attributes, subplots_pie, plot_attribute, plot_attribute2attribute
from project.utils import cumulated_plot,cumulated_plots, DECILES2QUINTILES


In [ ]:
output = os.path.join('data', 'derived', 'building_stock', 'img')
os.makedirs(output, exist_ok=True)


## Loading inputs
Inputs come from the Res-IRF reference scenario.


In [ ]:
stock = pd.read_csv(os.path.join('data', 'derived', 'building_stock', 'building_stock_sdes2018_aggregated.csv'), index_col=[0, 1, 2, 3, 4]).squeeze()


In [ ]:
# rename 'Energy performance' index to 'Performance'
quintile = True
stock.index.rename({'Energy performance':'Performance', 'Heating energy': 'Energy'}, inplace=True)
if quintile:
    stock.rename(index=DECILES2QUINTILES, level='Income tenant', inplace=True)
    resources_data['index']['Income tenant'] = ['C1', 'C2', 'C3', 'C4', 'C5']
    
dict_order  = {k: i for k, i in resources_data['index'].items() if k in stock.index.names}
for k in dict_order.keys():
    dict_order[k] = stock.index.get_level_values(k).unique()


## General description


In [ ]:
subplots_attributes(stock, dict_order=dict_order, dict_color=resources_data['colors'], percent=False, sharey=True)
subplots_attributes(stock, dict_order=dict_order, dict_color=resources_data['colors'], percent=False, sharey=True, save=os.path.join(output, 'stock_sdes2018_all.png'))


In [ ]:
subplots_pie(stock, dict_order=resources_data['index'], pie=['Housing type', 'Energy', 'Occupancy status'], dict_color=resources_data['colors'], percent=False)


In [ ]:
plot_attribute(stock, attribute='Performance', dict_order=resources_data['index'], percent=False, dict_color=resources_data['colors'], width=0.4,
               save=os.path.join(output, 'stock_sdes2018_energy_performance.png'), figsize=(8.0, 6.0))


In [ ]:
plot_attribute2attribute(stock, 'Performance', 'Energy', dict_order=dict_order, dict_color=resources_data['colors'])
plot_attribute2attribute(stock, 'Performance', 'Energy', dict_order=dict_order, dict_color=resources_data['colors'], percent=False, save=os.path.join(output, 'stock_sdes2018_dpe_energy.png'))


In [ ]:
plot_attribute2attribute(stock, 'Performance', 'Energy', dict_order=dict_order, dict_color=resources_data['colors'], percent=True)


### Relation occupation status - building performance


In [ ]:
for k in stock.index.names:
    if k != 'Performance':
        plot_attribute2attribute(stock, 'Performance', k, dict_order=dict_order, dict_color=resources_data['colors'], percent=True, save=os.path.join(output, 'stock_performance_{}.png').format(k.replace(' ', '_'.lower())))


In [ ]:
plot_attribute2attribute(stock, 'Performance', 'Occupancy status', dict_order=dict_order, dict_color=resources_data['colors'], percent=True, save=os.path.join(output, 'stock_occupation_performance.png'))
plot_attribute2attribute(stock, 'Performance', 'Occupancy status', dict_order=dict_order, dict_color=resources_data['colors'], percent=True)


### Relation income - building performance


In [ ]:
plot_attribute2attribute(stock, 'Performance', 'Income tenant', dict_order=dict_order, dict_color=resources_data['colors'], percent=True, save=os.path.join(output, 'stock_income_performance.png'))
plot_attribute2attribute(stock, 'Performance', 'Income tenant', dict_order=dict_order, dict_color=resources_data['colors'], percent=True)


### Table


Income by energy performance certificate


In [ ]:
stock.groupby(['Performance', 'Income tenant']).sum().unstack('Income tenant')


In [ ]:
df = (stock.groupby(['Performance', 'Income tenant']).sum().unstack('Income tenant').T / stock.groupby(['Performance', 'Income tenant']).sum().unstack('Income tenant').sum(axis=1)).T
df.loc[:, resources_data['index']['Income owner']].to_csv(os.path.join(output, 'stock_income_performance_table.csv'))


Energy by energy performance certificate


In [ ]:
stock.groupby(['Performance', 'Energy']).sum().unstack('Energy')


Energy by energy performance certificate (%)


In [ ]:
(stock.groupby(['Performance', 'Energy']).sum().unstack('Energy').T / stock.groupby(['Performance', 'Energy']).sum().unstack('Energy').sum(axis=1)).T
